In [2]:
# -*- coding: utf-8 -*-
# Author: Qinghua Liu <liu.11085@osu.edu>
# License: Apache-2.0 License

import pandas as pd
import numpy as np
import torch
import random, argparse, time, os, logging
from TSB_AD.evaluation.metrics import get_metrics
from TSB_AD.utils.slidingWindows import find_length_rank
from TSB_AD.model_wrapper import *
from TSB_AD.HP_list import Optimal_Multi_algo_HP_dict

# seeding
seed = 2024
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
np.random.seed(seed)
random.seed(seed)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

print("CUDA available: ", torch.cuda.is_available())
print("cuDNN version: ", torch.backends.cudnn.version())

# List of univariable models. 
# Removed Donut due to its internal error.
# Also not included: all transformer model (except moment_ft), as well as licensed models (NORMA, Series2Graph)
model_list = ['AnomalyTransformer','IForest', 'LOF', 'PCA', 'HBOS', 'OCSVM', 'MCD', 'KNN', 'KMeansAD', 'KShapeAD',
              'COPOD', 'CBLOF', 'EIF', 'RobustPCA', 'AutoEncoder', 'CNN', 'LSTMAD', 'TranAD', 
              'OmniAnomaly', 'USAD', 'FITS']

for model in model_list:
    try:
        ## ArgumentParser
        parser = argparse.ArgumentParser(description='Generating Anomaly Score')
        parser.add_argument('--dataset_dir', type=str, default=r'C:\Users\Kai\Documents\Time_Series_Anomaly_Detection\Time-Series-Anomaly-Detection-Seminar\TSB-AD\Datasets\TSB-AD-M')
        parser.add_argument('--file_list', type=str, default=r'C:\Users\Kai\Documents\Time_Series_Anomaly_Detection\Time-Series-Anomaly-Detection-Seminar\TSB-AD\Datasets\File_List\one_set_test_multi.csv')
        parser.add_argument('--score_dir', type=str, default='eval/score/multi/')
        parser.add_argument('--save_dir', type=str, default='eval/metrics/multi/')
        parser.add_argument('--no-save', action='store_false', dest='save', default=True, help='Disable saving')
        parser.add_argument('--AD_Name', type=str, default=model)

        args = parser.parse_args([])

        os.makedirs(args.score_dir, exist_ok=True)
        os.makedirs(args.save_dir, exist_ok=True)

        target_dir = os.path.join(args.score_dir, args.AD_Name)
        target_dir_metrics = os.path.join(args.save_dir, args.AD_Name)
        os.makedirs(target_dir, exist_ok = True)
        os.makedirs(target_dir_metrics, exist_ok = True)
        logging.basicConfig(filename=f'{target_dir}/000_run_{args.AD_Name}.log', level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

        file_list = pd.read_csv(args.file_list)['file_name'].values
        Optimal_Det_HP = Optimal_Multi_algo_HP_dict[args.AD_Name]
        print('Optimal_Det_HP: ', Optimal_Det_HP)

        write_csv = []
        for filename in file_list:
            if os.path.exists(target_dir+'/'+filename.split('.')[0]+'.npy'): continue
            print('Processing:{} by {}'.format(filename, args.AD_Name))

            file_path = os.path.join(args.dataset_dir, filename)
            df = pd.read_csv(file_path).dropna()
            data = df.iloc[:, 0:-1].values.astype(float)
            label = df['Label'].astype(int).to_numpy()
            # print('data: ', data.shape)
            # print('label: ', label.shape)

            feats = data.shape[1]
            slidingWindow = find_length_rank(data[:,0].reshape(-1, 1), rank=1)
            train_index = filename.split('.')[0].split('_')[-3]
            data_train = data[:int(train_index), :]

            start_time = time.time()

            if args.AD_Name in Semisupervise_AD_Pool:
                output = run_Semisupervise_AD(args.AD_Name, data_train, data, **Optimal_Det_HP)
            elif args.AD_Name in Unsupervise_AD_Pool:
                output = run_Unsupervise_AD(args.AD_Name, data, **Optimal_Det_HP)
            else:
                raise Exception(f"{args.AD_Name} is not defined")

            end_time = time.time()
            run_time = end_time - start_time

            if isinstance(output, np.ndarray):
                logging.info(f'Success at {filename} using {args.AD_Name} | Time cost: {run_time:.3f}s at length {len(label)}')
                np.save(f"{target_dir}/{args.AD_Name}_{filename.split('.')[0]}.npy", output)
            else:
                logging.error(f'At {filename}: '+output)

            ### whether to save the evaluation result
            if args.save:
                print("args.save is triggering correctly")
                try:
                    evaluation_result = get_metrics(output, label, slidingWindow=slidingWindow)
                    print('evaluation_result: ', evaluation_result)
                    list_w = list(evaluation_result.values())
                except Exception as e:
                    logging.error(f"Error calling get_metrics for {filename}: {e}")
                    logging.error(f"Output shape: {output.shape}, Label shape: {label.shape}, Sliding window: {slidingWindow}")
                    # Optionally log parts of the arrays if helpful, e.g.:
                    # logging.error(f"Output sample: {output[:10]}")
                    # logging.error(f"Label sample: {label[:10]}")
                    list_w = [0]*9
                list_w.insert(0, run_time)
                list_w.insert(0, filename)
                write_csv.append(list_w)

                ## Temp Save
                col_w = list(evaluation_result.keys())
                col_w.insert(0, 'Time')
                col_w.insert(0, 'file')
                w_csv = pd.DataFrame(write_csv, columns=col_w)
                w_csv.to_csv(f"{target_dir_metrics}/{args.AD_Name}.csv", index=False)
    except Exception as e:
        print(f"{model}_not working. Error: {e}")

CUDA available:  True
cuDNN version:  8801
Optimal_Det_HP:  {'win_size': 50, 'lr': 0.001}
Processing:057_SMD_id_1_Facility_tr_4529_1st_4629.csv by AnomalyTransformer
----- Using GPU NVIDIA GeForce RTX 3060 Ti -----
======================TRAIN MODE======================


Validation Epoch: : 100%|██████████| 7/7 [00:00<00:00, 31.38it/s, loss1=-44.5, loss2=45.4]


Updating learning rate to 0.001


Validation Epoch: : 100%|██████████| 7/7 [00:00<00:00, 31.89it/s, loss1=-46.6, loss2=47]  


EarlyStopping counter: 1 out of 7
Updating learning rate to 0.0005


...

evaluation_result:  {'AUC-PR': 0.5209520105013405, 'AUC-ROC': 0.9776350484652048, 'VUS-PR': 0.4727357099518004, 'VUS-ROC': 0.9525041403581878, 'Standard-F1': 0.5075364017742304, 'PA-F1': 0.6339410939691444, 'Event-based-F1': 0.5325310267017671, 'R-based-F1': 0.5607326995493985, 'Affiliation-F': 0.914438540058764}
Optimal_Det_HP:  {'win_size': 100, 'lr': 0.001}
Processing:057_SMD_id_1_Facility_tr_4529_1st_4629.csv by FITS
----- Using GPU NVIDIA GeForce RTX 3060 Ti -----


Testing: : 100%|██████████| 185/185 [00:01<00:00, 139.81it/s]


args.save is triggering correctly


c:\Users\Kai\anaconda3\envs\TSB-AD\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


evaluation_result:  {'AUC-PR': 0.08228709708958351, 'AUC-ROC': 0.8150332345583303, 'VUS-PR': 0.0984616857344197, 'VUS-ROC': 0.8464467937918657, 'Standard-F1': 0.14971289194963588, 'PA-F1': 0.7992733878292462, 'Event-based-F1': 0.28169014084507005, 'R-based-F1': 0.14781252910690573, 'Affiliation-F': 0.806731622620882}
